#### environment

In [1]:
import os 
import numpy as np
from numpy import full
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import MultipleLocator
from predict_loop_functions import predict_loop, noise_iterations, add_brain_region
from scipy.stats import linregress

In [2]:
# use image stack from july16 already saved
image_stack = np.load('saved_inputs_outputs//image_stack_july16.npy')

In [3]:
neurons = np.array([1685,368,1571,670,1918,1231,6795,6492,2507,1444,2361,6699,6672,6790,6209,6699,
           1447,5583,2237,2161,7217,5114,4321,4823,6822,5935,823,6501,6819,5782])

#### predictions

In [4]:
d3_sum, d3_mean, d3_var, d3_labels = predict_loop('dynamic', image_stack, 3, [[4,7]])

In [5]:
d15_sum, d15_mean, d15_var, d15_labels = predict_loop('dynamic', image_stack, 15, [[4,7]])

In [6]:
d30_sum, d30_mean, d30_var, d30_labels = predict_loop('dynamic', image_stack, 30, [[4,7]])

In [7]:
c3_sum, c3_mean, c3_var, c3_labels = predict_loop('constant', image_stack, 3, [[4,7]])

In [8]:
c15_sum, c15_mean, c15_var, c15_labels = predict_loop('constant', image_stack, 15, [[4,7]])

In [9]:
c30_sum, c30_mean, c30_var, c30_labels = predict_loop('constant', image_stack, 30, [[4,7]])

In [10]:
dbin_sum, dbin_mean, dbin_var, dbin_labels = predict_loop('dynamic', image_stack, 0, [[4,7]], stochastic_bin_param = True)

In [11]:
cbin_sum, cbin_mean, cbin_var, cbin_labels = predict_loop('constant', image_stack, 0, [[4,7]], stochastic_bin_param = True)

In [ ]:
# save predictions to predictions_july21
for noise in ['c3', 'c15', 'c30', 'd3', 'd15', 'd30', 'dbin', 'cbin']:
    for type in ['_sum', '_mean', '_var', '_labels']:
        file_name = noise + type
        path = 'saved_inputs_outputs//predictions_july21//' + file_name
        np.save(path, arr = globals()[file_name])

#### load predictions

In [78]:
for noise in ['c3', 'c15', 'c30', 'd3', 'd15', 'd30', 'dbin', 'cbin']:
    for type in ['_sum', '_mean', '_var', '_labels']:
        file_name = noise + type + '.npy'
        var_name = noise + type
        path = 'saved_inputs_outputs//predictions_july21//' + file_name
        globals()[var_name] = np.load(path)

#### filtering by region

In [59]:
c3_labels_test = c3_labels

In [60]:
c3_labels_test = c3_labels_test.reshape(c3_labels_test.shape[1],)

In [64]:
c3_labels_test[:5]

array([2, 2, 2, 2, 2])

In [40]:
c3_mean.shape

(5, 7493)

In [38]:
print('Brain Region Labels Value Counts\n' \
      '--------------------------------')
print(pd.DataFrame(c3_labels).T.value_counts())

Brain Region Labels Value Counts
--------------------------------
0
1    5312
2     983
4     811
3     387
Name: count, dtype: int64


In [72]:
# make each region 100 neurons
region3 = []

for i, region in enumerate(c3_labels_test):
    if region == 3: region3.append([i, int(region)])

In [74]:
region3

[[513, 3],
 [514, 3],
 [515, 3],
 [900, 3],
 [940, 3],
 [1000, 3],
 [1001, 3],
 [1718, 3],
 [2101, 3],
 [2170, 3],
 [2173, 3],
 [2195, 3],
 [2197, 3],
 [2218, 3],
 [2219, 3],
 [2220, 3],
 [2221, 3],
 [2222, 3],
 [2223, 3],
 [2224, 3],
 [2225, 3],
 [2226, 3],
 [2227, 3],
 [2228, 3],
 [2229, 3],
 [2230, 3],
 [2231, 3],
 [2232, 3],
 [2233, 3],
 [2234, 3],
 [2235, 3],
 [2236, 3],
 [2237, 3],
 [2238, 3],
 [2239, 3],
 [2240, 3],
 [2241, 3],
 [2242, 3],
 [2245, 3],
 [2247, 3],
 [2248, 3],
 [2249, 3],
 [2250, 3],
 [2251, 3],
 [2252, 3],
 [2253, 3],
 [2255, 3],
 [2256, 3],
 [2257, 3],
 [2258, 3],
 [2259, 3],
 [2260, 3],
 [2261, 3],
 [2262, 3],
 [2266, 3],
 [2267, 3],
 [2268, 3],
 [2269, 3],
 [2270, 3],
 [2271, 3],
 [2272, 3],
 [2273, 3],
 [2274, 3],
 [2275, 3],
 [2276, 3],
 [2277, 3],
 [2278, 3],
 [2279, 3],
 [2280, 3],
 [2281, 3],
 [2291, 3],
 [2292, 3],
 [2293, 3],
 [2294, 3],
 [2295, 3],
 [2306, 3],
 [2307, 3],
 [2308, 3],
 [2309, 3],
 [2312, 3],
 [2850, 3],
 [2858, 3],
 [2885, 3],
 [2906, 3

In [75]:
region3_indices = [i for i, region in region3]

In [76]:
region3_mean = c3_mean[:, region3_indices]

In [77]:
region3_mean.shape

(5, 387)

shape matches value_counts() for region

#### sigma against mean

In [17]:
c3_sum.shape

(5, 100, 7493)

In [18]:
c3_mean.shape

(5, 7493)

In [25]:
def line_plot_mean(array, neurons): # input mean output from predict_loop()
    final_array = np.empty(shape = len(neurons))
    for i, neuron in enumerate(neurons):
        avg = np.mean(array[:,neuron])
        final_array[i] = avg
    return final_array

In [30]:
plot_mean_d30 = line_plot_mean(d30_sum, [i for i in range(100)])
plot_mean_d15 = line_plot_mean(d15_sum, [i for i in range(100)])
plot_mean_d3 = line_plot_mean(d3_sum, [i for i in range(100)])
plot_mean_c30 = line_plot_mean(c30_sum, [i for i in range(100)])
plot_mean_c15 = line_plot_mean(c15_sum, [i for i in range(100)])
plot_mean_c3 = line_plot_mean(c3_sum, [i for i in range(100)])